In [1]:
from base import generator_haar

def init_experiment(n):
    d = 2**n

    # Generate 6^n density matrices
    rho_list = generator_haar.generate_n_qubits_rho_haar(n)
    print(f"Generated {len(rho_list)} of {rho_list[0].shape} rho.")

    # Generate unitary
    unitary = generator_haar.random_unitary(d)
    print(f"Generated {unitary.shape} unitary operators.")
    return rho_list, unitary

2025-03-23 13:26:15.404797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-23 13:26:15.405547: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-23 13:26:15.409455: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-23 13:26:15.423024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742703975.446640  404633 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742703975.45

In [2]:
import numpy as np
import tensorflow as tf

from base import epsilon_rho
def calculate_rho2_unitary(rho_list, unitary):
    rho2_unitary = []
    for rho in rho_list:
        rho2_unitary.append(epsilon_rho.calculate_from_unitary(rho, unitary))
    return rho2_unitary

def calculate_rho2_dephasing(rho_list, n, gamma):
    rho2 = []
    for rho in rho_list:
        rho2.append(epsilon_rho.calculate_dephasing(rho, n, gamma))
    return rho2

def write_to_file(filename, data):
    """Write TensorFlow tensor data to a text file without truncation."""
    tensor_data = data.numpy() if isinstance(data, tf.Tensor) else data

    # Open the file and write the tensor data
    with open(filename, 'w') as f:
        if isinstance(data, np.ndarray):
            np.savetxt(f, data, fmt="%.6f")
        elif isinstance(data, list):
            for item in data:
                f.write(f"{item}\n")
        else:
            f.write(str(data))





In [3]:
import os
from base import optimize_algorithm
from base import metrics
experiment_folder = ''

for num_qubits in range(1, 2):
    if (experiment_folder == ''):
        break
    else:
        write_folder = os.path.join(experiment_folder, str(num_qubits) + "_qubits")
        if not os.path.exists(write_folder):
            os.makedirs(write_folder)
    print(f"N={num_qubits}")

    #-----Init experiment-----
    rho_list, unitary = init_experiment(num_qubits)
    write_to_file(os.path.join(write_folder, "rho_list.txt"), rho_list)

    g_s = np.linspace(1, 10e-3, 20)
    for g in g_s:
        folder_path = os.path.join(write_folder, "_{:.2f}".format(g))
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        rho2_list = calculate_rho2_dephasing(rho_list, num_qubits, g)
    
        #-----Learn kraus operators-----
        unitary_res, cost_dict = optimize_algorithm.optimize_adam_unitary_dagger_set(rho_list, rho2_list, unitary, 0.008, num_loop=200)
    
        #-----Calculate result data-----
        rho3_list = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res, rho2_list)
        rho2_unitary_list = calculate_rho2_unitary(rho_list, unitary_res)
    
        mean_fidelity_rho_rho3 = metrics.mean_fidelity(rho3_list, rho_list)
        mean_fidelity_rho2_rho2 = metrics.mean_fidelity(rho2_unitary_list, rho2_list)

        #-----Write to folder-----    
        write_to_file(os.path.join(folder_path,"unitary.txt"), unitary)
        write_to_file(os.path.join(folder_path,"unitary_res.txt"), unitary_res)
        write_to_file(os.path.join(folder_path,"cost_dict.txt"), cost_dict)

        write_to_file(os.path.join(folder_path,"rho2_list.txt"), rho2_list)
        write_to_file(os.path.join(folder_path,"rho2_unitary_list.txt"), rho2_unitary_list)

        write_to_file(os.path.join(folder_path,"mean_fidelity_rho_rho3.txt"), mean_fidelity_rho_rho3.numpy())
        write_to_file(os.path.join(folder_path,"mean_fidelity_rho2_rho2.txt"), mean_fidelity_rho2_rho2.numpy())

        print(g, num_qubits)
        print(cost_dict[-1])

    
    

In [4]:
import tensorflow as tf
import numpy as np
import os
import re
import matplotlib.pyplot as plt

def parse_tensor_from_file(file_path, shape):
    with open(file_path, 'r') as file:
        # Read the content of the file
        tensor_str = file.read()

    components = re.findall(r"([+-]?\d+\.?\d*[eE]?[+-]?\d*(?:\s*[+-]?\d*\.?\d*[eE]?[+-]?\d*j)?)", tensor_str)
    # Convert the components into a numpy array of complex numbers
    complex_numbers = [complex(c.replace(' ','')) for c in components]

    # Convert the list to a NumPy array and reshape it
    numpy_tensor = np.array(complex_numbers, dtype=np.complex128)

    if (shape == 3):
        numpy_tensor = numpy_tensor.reshape(round(numpy_tensor.size ** (1/shape)),round(numpy_tensor.size ** (1/shape)),round(numpy_tensor.size ** (1/shape)))
    if (shape == 2):
        size = numpy_tensor.size
        new_shape = (round(size ** (1/shape)), round(size ** (1/shape))) if shape == 2 else (size,)
        numpy_tensor = numpy_tensor.reshape(new_shape)
    # Convert NumPy array to TensorFlow tensor
    tf_tensor = tf.convert_to_tensor(numpy_tensor)

    return tf_tensor

experiment_folder = 'results/experiment_new/dephasing/5_qubits'

num_qubits = 5
rho_list_test = generator_haar.generate_n_qubits_rho_haar(num_qubits)
print(f"Generated {len(rho_list_test)} of {rho_list_test[0].shape} rho.")
write_to_file(os.path.join(experiment_folder, "rho_list_test.txt"), rho_list_test)

g_s = np.linspace(1, 10e-3, 20)
for g in g_s:
    folder_path = os.path.join(experiment_folder, "_{:.2f}".format(g))

    rho2_list = calculate_rho2_dephasing(rho_list_test, num_qubits, g)
    
    #-----Learn kraus operators-----
    unitary_res = parse_tensor_from_file(os.path.join(folder_path, "unitary_res.txt"), 2)
    
    #-----Calculate result data-----
    rho3_list = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res, rho2_list)
    rho2_unitary_list = calculate_rho2_unitary(rho_list_test, unitary_res)
    
    mean_fidelity_rho_rho3 = metrics.mean_fidelity(rho3_list, rho_list_test)
    mean_fidelity_rho2_rho2 = metrics.mean_fidelity(rho2_unitary_list, rho2_list)

    #-----Write to folder-----    
    write_to_file(os.path.join(folder_path,"rho2_list_test.txt"), rho2_list)
    write_to_file(os.path.join(folder_path,"rho2_unitary_list_test.txt"), rho2_unitary_list)

    write_to_file(os.path.join(folder_path,"mean_fidelity_rho_rho3_test.txt"), mean_fidelity_rho_rho3.numpy())
    write_to_file(os.path.join(folder_path,"mean_fidelity_rho2_rho2_test.txt"), mean_fidelity_rho2_rho2.numpy())

    print(g, mean_fidelity_rho_rho3)

Generated 7776 of (32, 32) rho.


W0000 00:00:1742704005.324648  404633 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


1.0 tf.Tensor((0.05853790974957586-7.778205494685249e-12j), shape=(), dtype=complex128)
0.9478947368421052 tf.Tensor((0.11151755543056335-8.930645855562523e-12j), shape=(), dtype=complex128)
0.8957894736842105 tf.Tensor((0.14762029124213752+4.44315458029875e-11j), shape=(), dtype=complex128)
0.8436842105263158 tf.Tensor((0.18390556202148067+2.2305192640269375e-12j), shape=(), dtype=complex128)
0.791578947368421 tf.Tensor((0.22227750904979254-7.319141123082271e-11j), shape=(), dtype=complex128)
0.7394736842105263 tf.Tensor((0.25977614745207933+4.9009306692641406e-11j), shape=(), dtype=complex128)
0.6873684210526316 tf.Tensor((0.29667668008741177-1.6519406866228085e-11j), shape=(), dtype=complex128)
0.6352631578947369 tf.Tensor((0.34110700346801753+3.711392585715068e-11j), shape=(), dtype=complex128)
0.5831578947368421 tf.Tensor((0.3815090102010744-1.9220597764196885e-11j), shape=(), dtype=complex128)
0.5310526315789473 tf.Tensor((0.4214815949189668-6.516651998386627e-11j), shape=(), dty

In [10]:
import os
from base import metrics
experiment_folder = 'results/experiment_new/dephasing_matrix_comparison'
#experiment_folder = 'results/experiment_new/haar_random_matrix_comparison'
rho_test = generator_haar.generate_rho_haar(1)

rho2_001 = epsilon_rho.calculate_dephasing(rho_test, 1, 0.01)

rho2_1 = epsilon_rho.calculate_dephasing(rho_test, 1, 1)

unitary_001 = np.array([
    [-0.999974 + 0.007165j, -0.000163 - 0.000430j],
    [ 0.000157 - 0.000432j, -0.999975 + 0.007117j]
])
unitary_1 = np.array([
    [-0.973137 + 0.147438j, -0.079068 - 0.158163j],
    [ 0.072494 - 0.161282j, -0.978378 - 0.107277j]
])
rho2_unitary_001 = epsilon_rho.calculate_from_unitary(rho2_001, unitary_001)

rho2_unitary_1 = epsilon_rho.calculate_from_unitary(rho2_1, unitary_1)

if (experiment_folder!=''):
    write_to_file(os.path.join(experiment_folder,"rho.txt"), rho_test)
    write_to_file(os.path.join(experiment_folder,"rho2_001.txt"), rho2_001)
    write_to_file(os.path.join(experiment_folder,"rho2_1.txt"), rho2_1)
    write_to_file(os.path.join(experiment_folder,"rho2_unitary_001.txt"), rho2_unitary_001)
    write_to_file(os.path.join(experiment_folder,"rho2_unitary_1.txt"), rho2_unitary_1)
    write_to_file(os.path.join(experiment_folder,"fide001.txt"), metrics.compilation_trace_fidelity(rho2_001, rho2_unitary_001))
    write_to_file(os.path.join(experiment_folder,"fide1.txt"), metrics.compilation_trace_fidelity(rho2_1, rho2_unitary_1))
    